In [1]:
import planetary_computer as pc
import pystac_client
import stackstac
import rioxarray
import geopandas as gpd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from ipyleaflet import Map, DrawControl, basemaps, LayersControl
from ipywidgets import VBox, Button, Output
import shapely.geometry as geom
import xarray as xr
from pystac.extensions.eo import EOExtension as eo
from dask_gateway import GatewayCluster

In [5]:
from dask.distributed import Client, progress
client = Client(processes=False, threads_per_worker=4,
                n_workers=1, memory_limit='2GB')
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://172.21.36.52:8787/status,
Dashboard: http://172.21.36.52:8787/status,Workers: 1
Total threads: 4,Total memory: 1.86 GiB
Status: running,Using processes: False
Comm: inproc://172.21.36.52/27509/1,Workers: 0
Dashboard: http://172.21.36.52:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: inproc://172.21.36.52/27509/4,Total threads: 4
Dashboard: http://172.21.36.52:42097/status,Memory: 1.86 GiB
Nanny: None,


In [6]:
cluster = GatewayCluster()  # Creates the Dask Scheduler. Might take a minute.

client = cluster.get_client()

cluster.adapt(minimum=4, maximum=24)
print(cluster.dashboard_link)

ValueError: No dask-gateway address provided or found in configuration

In [3]:
m = Map(center=(-7,10), zoom=2, basemap=basemaps.Esri.WorldImagery)
draw_control = DrawControl(polyline={}, circle={}, circlemarker={}, marker={},
                           polygon={"shapeOptions": {"color": "#6bc2e5"}})
aoi_geojson = []
def handle_draw(self, action, geo_json):
    if action == 'created':
        aoi_geojson.clear()
        aoi_geojson.append(geo_json['geometry'])
        print("AOI captured.")
draw_control.on_draw(handle_draw)
m.add_control(draw_control)
m.add_control(LayersControl())
display(m)

print("👉 Draw your AOI polygon above, then run the next cell.")

Map(center=[-7, 10], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_tex…

👉 Draw your AOI polygon above, then run the next cell.


In [5]:
# After drawing AOI run this cell
from shapely.geometry import shape
assert aoi_geojson, "⚠️ Please draw an AOI first!"
aoi_shape = shape(aoi_geojson[0])
aoi = gpd.GeoSeries([aoi_shape], crs="EPSG:4326")
bbox = aoi.total_bounds.tolist()  # [minx,miny,maxx,maxy]
print("AOI bounds:", bbox)

AOI bounds: [-16.640836, 14.111538, -16.233044, 14.502763]


In [7]:
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=pc.sign_inplace,
)

In [ ]:
# 🎯 Sentinel-2 band asset names (12 bands, excluding SCL and QA)
# bands_12 = [
#     "B01", "B02", "B03", "B04", "B05", "B06", "B07",
#     "B08", "B8A", "B09", "B11", "B12"
# ]

def search_s2(year: int, season: str):
    if season == "wet":
        start = f"{year}-06-01"
        end = f"{year}-10-15"
    else:  # dry season
        start = f"{year}-10-16"
        end = f"{year+1}-05-31"

    # STAC search
    search = catalog.search(
        collections=["sentinel-2-l2a"],
        bbox=bbox,
        datetime=f"{start}/{end}",
        query={"eo:cloud_cover": {"lt": 30}},
        limit=100,
    )
    items = list(search.get_items())
    print("Items found:", len(items))
    if items:
        print("Example item assets:", list(items[0].assets.keys()))
    if not items:
        print(f"No Sentinel-2 items found for {year}-{season}")
        return None

    # Sign for Planetary Computer access
    signed_items = [pc.sign(item).to_dict() for item in items]

    assets = ["B02", "B03", "B04", "B08"]  # test subset

    data = stackstac.stack(
        signed_items,
        assets=assets,
        epsg=4326,
    # resolution=10,   # comment out for now
    # bounds_latlon=bbox,  # comment out for now
)
    print("✅ Stack shape:", data.shape)
    print("✅ Dimensions:", data.dims)
    print("✅ Coords:", list(data.coords))

# filter nodata
    data = data.where(data > 0, np.nan)


    # Persist to speed up median calculation
    data = data.persist()
    median = data.median(dim="time").compute()
    return median  # shape: (band, y, x)

# ------------------------------
# 📦 Build seasonal composites
# ------------------------------
years = [2022]
seasonals = []

for y in years:
    for s in ["wet", "dry"]:
        print(f"⏳ Creating composite for {y}-{s} …")
        comp = search_s2(y, s)
        if comp is not None:
            seasonals.append({
                "year": y,
                "season": s,
                "image": comp
            })

print("✅ Finished creating composites!")
print(f"Total composites: {len(seasonals)}")

# Example: access the first composite
first_comp = seasonals[0]["image"]
print(first_comp)
print("Bands:", first_comp.band.values)
print("Shape:", first_comp.shape)  # (bands, y, x)


⏳ Creating composite for 2022-wet …


/mnt/d/PHD/PLUS-CLIMB/Notebooks/planetary-classification/.venv/lib/python3.12/site-packages/pystac_client/item_search.py:925: FutureWarning: get_items() is deprecated, use items() instead
  warnings.warn(


Items found: 16
Example item assets: ['AOT', 'B01', 'B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B09', 'B11', 'B12', 'B8A', 'SCL', 'WVP', 'visual', 'safe-manifest', 'granule-metadata', 'inspire-metadata', 'product-metadata', 'datastrip-metadata', 'tilejson', 'rendered_preview']
✅ Stack shape: (16, 4, 20928, 11067)
✅ Dimensions: ('time', 'band', 'y', 'x')
✅ Coords: ['time', 'id', 'band', 'x', 'y', 'instruments', 's2:generation_time', 's2:snow_ice_percentage', 's2:mean_solar_azimuth', 's2:vegetation_percentage', 'proj:code', 's2:datatake_id', 's2:processing_baseline', 's2:mgrs_tile', 's2:not_vegetated_percentage', 'eo:cloud_cover', 's2:cloud_shadow_percentage', 's2:thin_cirrus_percentage', 'constellation', 's2:degraded_msi_data_percentage', 's2:dark_features_percentage', 's2:product_type', 's2:saturated_defective_pixel_percentage', 's2:reflectance_conversion_factor', 's2:water_percentage', 's2:datatake_type', 'sat:orbit_state', 's2:unclassified_percentage', 'platform', 'sat:relative